In [ ]:
!pip install -U diffusers transformers accelerate torch torchvision -q

In [ ]:
!pip uninstall -y torchaudio -q

In [2]:
import torch, gc, json, requests
from diffusers import FluxPipeline, StableDiffusionXLPipeline, StableDiffusion3Pipeline

url = "https://raw.githubusercontent.com/maram-elaian/brandora/main/data/test_briefs.json"
briefs = requests.get(url).json()[:5]   # 5 بس للتجربة الأولى، مش الـ30 كاملة

def brief_to_image_prompt(b):
    return f"{b['logo_direction']}, {', '.join(b['visual_style'])} style, vector logo, clean background, no text"

models_to_check = [
    {"name": "FLUX_schnell", "id": "black-forest-labs/FLUX.1-schnell", "cls": FluxPipeline, "dtype": torch.bfloat16, "steps": 4, "guidance": 0.0},
    {"name": "Playground_v2.5", "id": "playgroundai/playground-v2.5-1024px-aesthetic", "cls": StableDiffusionXLPipeline, "dtype": torch.float16, "steps": 25, "guidance": 3.0},
    {"name": "SD3_medium", "id": "stabilityai/stable-diffusion-3-medium-diffusers", "cls": StableDiffusion3Pipeline, "dtype": torch.float16, "steps": 28, "guidance": 7.0},
]

for m in models_to_check:
    print(f"⏳ جاري تحميل: {m['name']}")
    pipe = None
    try:
        pipe = m["cls"].from_pretrained(m["id"], torch_dtype=m["dtype"])
        pipe.enable_model_cpu_offload()

        for brief in briefs:
            prompt = brief_to_image_prompt(brief)
            image = pipe(
                prompt,
                num_inference_steps=m["steps"],
                guidance_scale=m["guidance"],
            ).images[0]
            filename = f"{m['name']}_{brief['id']}.png"
            image.save(filename)
            print(f"   ✅ {brief['id']} → {filename}")

    except Exception as e:
        print(f"   ❌ فشل! {type(e).__name__}: {e}")

    finally:
        if pipe is not None:
            del pipe
        gc.collect()
        torch.cuda.empty_cache()

print("✅ انتهى!")

⏳ جاري تحميل: FLUX_schnell
   ❌ فشل! GatedRepoError: 401 Client Error. (Request ID: Root=1-6ab4e033-346968ec0f13de035145762c;f79d3052-db16-4d78-ba6b-a98315d269e9)

Cannot access gated repo for url https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/model_index.json.
Access to model black-forest-labs/FLUX.1-schnell is restricted. You must have access to it and be authenticated to access it. Please log in.
⏳ جاري تحميل: Playground_v2.5


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

...

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


   ✅ BR004 → Playground_v2.5_BR004.png


  0%|          | 0/25 [00:00<?, ?it/s]

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


   ✅ BR005 → Playground_v2.5_BR005.png
⏳ جاري تحميل: SD3_medium
   ❌ فشل! GatedRepoError: 401 Client Error. (Request ID: Root=1-6ab4e0eb-6f16627a6e5571e14b4fe7be;55302d93-c81c-4f82-af7f-e3cf517c1d72)

Cannot access gated repo for url https://huggingface.co/stabilityai/stable-diffusion-3-medium-diffusers/resolve/main/model_index.json.
Access to model stabilityai/stable-diffusion-3-medium-diffusers is restricted. You must have access to it and be authenticated to access it. Please log in.
✅ انتهى!


In [6]:
# الخلية 1: تعريف المتغيّر (لازم تشتغل أول، بنفس الجلسة)
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
print("تم تحميل التوكن ✅")

تم تحميل التوكن ✅


In [7]:
# الخلية 2: فحص الوصول (تشتغل بعدها مباشرة)
from huggingface_hub import HfApi, login

login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)

try:
    info = api.model_info("black-forest-labs/FLUX.1-schnell")
    print("✅ عندك وصول فعلي لـFLUX.1-schnell")
except Exception as e:
    print(f"❌ لسا ما في وصول: {e}")

✅ عندك وصول فعلي لـFLUX.1-schnell
